# Unsupervised bearing fault detection under operating-condition shift

Runs the pipeline in [`src/`](https://github.com/) against a CWRU dataset mounted from Kaggle.

**Order of operations**
1. Push this repository to GitHub first (the code is the deliverable, not the notebook).
2. Add a CWRU `.mat` dataset to this notebook via *Add Data* (e.g. search `CWRU .mat`).
3. Set `REPO_URL` below, run all, then copy `results/` back into the repo and commit it.

Internet must be enabled in the notebook settings for the `git clone` step.


In [ ]:
REPO_URL = 'https://github.com/<your-username>/bearing-ae.git'  # <-- set this
CONFIG   = 'configs/dense_logspec.yaml'

import os, subprocess, sys
os.chdir('/kaggle/working')
if not os.path.isdir('bearing-ae'):
    subprocess.run(['git', 'clone', '--quiet', REPO_URL, 'bearing-ae'], check=True)
os.chdir('/kaggle/working/bearing-ae')
print(subprocess.run(['git','rev-parse','--short','HEAD'], capture_output=True, text=True).stdout.strip())


## 1. Inspect what the mounted dataset actually contains

Mirrors differ in layout and file naming. Look before configuring anything —
in particular check whether 48 kHz recordings are mixed in with the 12 kHz ones.


In [ ]:
from pathlib import Path
from collections import Counter

mats = sorted(Path('/kaggle/input').rglob('*.mat'))
print(f'{len(mats)} .mat files under /kaggle/input')
for p in mats[:15]:
    print(' ', p)
print('...')
print(Counter(p.parent.as_posix() for p in mats).most_common(10))


## 2. Point the config at the data

Everything that affects a result stays in the config file, including the seed.
This cell writes a run-specific copy rather than editing the versioned one.


In [ ]:
import yaml

cfg = yaml.safe_load(open(CONFIG))
cfg['data']['roots'] = ['/kaggle/input']
cfg['data']['exclude'] = ['48k']   # adjust after looking at the listing above
cfg['data']['include'] = []        # e.g. ['12k'] if the mirror is organised that way

run_cfg = '/kaggle/working/config_run.yaml'
yaml.safe_dump(cfg, open(run_cfg, 'w'), sort_keys=False)
print(open(run_cfg).read())


## 3. Train and evaluate

Healthy windows at one motor load only. The threshold comes from a held-out
healthy split at that same load; fault labels are used for scoring and nothing else.


In [ ]:
!python -m src.main --config {run_cfg} --out results


## 4. Results


In [ ]:
import json
from IPython.display import Image, display

metrics = json.load(open('results/metrics.json'))
print(json.dumps({k: v for k, v in metrics.items() if k != 'config'}, indent=2))

for fig in ['fpr_by_load.png', 'scores_train_load.png', 'scores_shifted_loads.png', 'training_curve.png']:
    path = Path('results') / fig
    if path.exists():
        display(Image(str(path)))


## 5. Read the shift result before writing anything down

The number that matters is `false_positive_rate_by_load`. If it is flat, the
representation is load-invariant and the detector transfers. If it climbs away
from the training load, the threshold does not transfer — which is the normal
outcome, and the finding worth reporting rather than hiding.

Compare against `kurtosis_baseline`. If the scalar baseline degrades less than
the autoencoder under shift, say so in the README. That comparison is the most
credible thing in the repository.


In [ ]:
!zip -qr /kaggle/working/results.zip results && echo 'download results.zip, commit it under results/ in the repo'
